#Multilingual Mobile App Review-**2025**




Link for the uploaded dataset from Kaggle : https://www.kaggle.com/code/babajannarmedov/multilingual-mobile-app-review-2025

In [ ]:
import io, os, pandas as pd
import numpy as np



# 1. DATA COLLECTION

In [ ]:
df = pd.read_csv("/content/multilingual_mobile_app_reviews_2025.csv")
df.head()

In [ ]:
df.describe()

# save a local copy in the working directory

In [ ]:
df.to_csv("2470231_DA_LAB_Assignment.csv", index=False)

In [ ]:
import pandas as pd

# Path to your CSV file
input_path = '/content/multilingual_mobile_app_reviews_2025.csv'

# Load the CSV into a pandas DataFrame
df = pd.read_csv(input_path)

# Display confirmation and shape
print(f"Data loaded successfully from: {input_path}")
print("Shape of the DataFrame:", df.shape)

# Optionally save a local copy (not necessary in this case, but kept for completeness)
output_path = 'multilingual_mobile_app_reviews_2025_copy.csv'



In [ ]:
# Install ydata-profiling in Colab
!pip -q install ydata-profiling


# 2. REPORT GENERATION

In [ ]:
from ydata_profiling import ProfileReport

# Create a lightweight report (good for big datasets)
profile = ProfileReport(df, title="Dataset Profile Report", minimal=True, explorative=True)

# Save to HTML
report_path = "profile_report.html"
profile.to_file(report_path)

# Display inside the notebook (optional)
from IPython.display import IFrame, display
display(IFrame(src=report_path, width="100%", height=800))


In [ ]:
#since the html display had issues, I followed up with this codec for the optimal solution.

from IPython.display import HTML

with open("profile_report.html", "r", encoding="utf-8") as f:
    html = f.read()

display(HTML(html))


# 3. DATA PREPROCESSING

In [ ]:
# 4) Basic preprocessing
import numpy as np

print("Initial shape:", df.shape)

# 4.1 Remove exact duplicate rows
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Duplicates removed: {before-after}. Current shape: {df.shape}")

# 4.2 Detect missing values
miss = df.isnull().sum()
print("Missing values per column (showing non-zero only):")
print(miss[miss>0])

# 4.3 Simple missing-value strategy:
# - For numeric: fill with median
# - For object/categorical: fill with mode (if exists) or 'missing'
for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            med = df[col].median()
            df[col] = df[col].fillna(med)
            print(f"Filled numeric column '{col}' with median = {med}")
        else:
            try:
                mode = df[col].mode(dropna=True)[0]
            except IndexError:
                mode = "missing"
            df[col] = df[col].fillna(mode)
            print(f"Filled non-numeric column '{col}' with mode = {mode}")

# 4.4 Convert obvious numeric-looking strings to numbers
for col in df.select_dtypes(include=['object']).columns:
    # try convert
    coerced = pd.to_numeric(df[col], errors='coerce')
    non_na = coerced.notna().sum()
    if non_na > 0 and non_na / len(df) > 0.5:
        df[col] = coerced
        print(f"Coerced column '{col}' to numeric (partial conversion).")

print("Preprocessing done. Shape:", df.shape)

In [ ]:
# 4.5 Label encoding for categorical variables (if any)
from sklearn.preprocessing import LabelEncoder
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
le_map = {}
for c in cat_cols:
    le = LabelEncoder()
    df[c] = df[c].astype(str)
    df[c] = le.fit_transform(df[c])
    le_map[c] = list(le.classes_)
    print(f"Label-encoded column '{c}' (classes: {len(le.classes_)})")

print("Label encoding finished. Current dtypes:\n", df.dtypes)

# 4. DESCRIPTIVE STATISTICS



In [ ]:
# 5) Descriptive statistics: skewness and simple outlier detection
import scipy.stats as stats
num_cols = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric columns detected:", num_cols)

skews = df[num_cols].skew()
print("\nSkewness (showing absolute skew > 1):")
print(skews[skews.abs() > 1])

# Simple outlier detection using IQR
outlier_summary = {}
for c in num_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    outliers = df[(df[c] < lower) | (df[c] > upper)][c]
    outlier_summary[c] = {'num_outliers': int(outliers.shape[0]), 'pct_outliers': float(outliers.shape[0]) / len(df) * 100}
print("\nOutlier summary (column: # outliers, %):")
for k,v in outlier_summary.items():
    if v['num_outliers']>0:
        print(k, v)

# 5. DATA TRANSFORMATIONS

In [ ]:
# 6) Data transformations: Scaling (StandardScaler) and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df.copy()

# Automatic target detection: common names or last column
candidate_targets = ['target','label','class','y']
target_col = None
for t in candidate_targets:
    if t in X.columns:
        target_col = t
        break
if target_col is None:
    # choose last column as target
    target_col = X.columns[-1]
print("Auto-detected target column:", target_col)

y = X[target_col]
X = X.drop(columns=[target_col])

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_features] = scaler.fit_transform(X[numeric_features])
print("Scaling applied to numeric features.")

# PCA (reduce to min(10, n_features))
n_comp = min(10, len(numeric_features))
if n_comp >= 1:
    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(X_scaled[numeric_features])
    print(f"PCA applied: explained variance ratio (first {min(5,n_comp)} components):", pca.explained_variance_ratio_[:min(5,n_comp)])
else:
    X_pca = None
    print("No numeric features available for PCA.")

In [ ]:
# 7) Predictive analysis: decide classification vs regression
import numpy as np
from sklearn.model_selection import train_test_split

y_unique = y.dropna().unique()
n_unique = len(y_unique)
is_classification = False
if pd.api.types.is_integer_dtype(y) or pd.api.types.is_object_dtype(y) or n_unique <= 20:
    is_classification = True
else:
    # if float but has few distinct values, still classification
    if n_unique <= 20:
        is_classification = True

print("Detected as classification?" , is_classification, "| # unique target values:", n_unique)

# Prepare X for modeling: use scaled numeric + remaining encoded features
X_model = X_scaled.copy()

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X_model, y, test_size=0.2, random_state=42)
print("Train/test shapes:", X_train.shape, X_test.shape)

# 6. PERFORMANCE ANALYSIS

In [ ]:
# 7.1 Train four models and evaluate
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
results = {}

if is_classification:
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier

    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'SVC': SVC(probability=True),
        'KNN': KNeighborsClassifier()
    }

    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            cm = confusion_matrix(y_test, preds)
            results[name] = {'accuracy': float(acc), 'confusion_matrix': cm.tolist()}
            print(f"{name}: accuracy={acc:.4f}")
        except Exception as e:
            results[name] = {'error': str(e)}
            print(f"{name}: ERROR ->", e)

else:
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.svm import SVR
    from sklearn.neighbors import KNeighborsRegressor

    models = {
        'LinearRegression': LinearRegression(),
        'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=42),
        'SVR': SVR(),
        'KNNRegressor': KNeighborsRegressor()
    }

    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            rmse = mean_squared_error(y_test, preds, squared=False)
            r2 = r2_score(y_test, preds)
            results[name] = {'rmse': float(rmse), 'r2': float(r2)}
            print(f"{name}: RMSE={rmse:.4f}, R2={r2:.4f}")
        except Exception as e:
            results[name] = {'error': str(e)}
            print(f"{name}: ERROR ->", e)

print('\nSummary results dictionary:')
results

Above rendered data
such results because of the chosen dataset from Kaggle.

In [ ]:
# 8) Save cleaned dataset and results
cleaned_path = "cleaned_data.csv"
df.to_csv(cleaned_path, index=False)
print("Cleaned dataset saved to", cleaned_path)

import json
with open("model_results.json","w") as f:
    json.dump(results, f)
print("Model results saved to model_results.json")

print("\nFiles created in working dir:")
!ls -la

END.